In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/brandao/diabetes/description.pdf
/kaggle/input/datasets/brandao/diabetes/diabetic_data.csv


In [2]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/brandao/diabetes/diabetic_data.csv')

In [3]:
import sqlite3
conn = sqlite3.connect(':memory:')
df.to_sql('admissions', conn, index=False)

101766

In [4]:
query = """
SELECT readmitted, AVG(time_in_hospital) as avg_stay, COUNT(*) as num_patients
FROM admissions
GROUP BY readmitted
"""
result = pd.read_sql(query, conn)
print(result)

  readmitted  avg_stay  num_patients
0        <30  4.768249         11357
1        >30  4.495541         35545
2         NO  4.254429         54864


In [5]:
query2 = """
SELECT age, COUNT(*) as total, 
SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) as readmitted_under_30
FROM admissions
GROUP BY age
ORDER BY age
"""
print(pd.read_sql(query2, conn))

        age  total  readmitted_under_30
0    [0-10)    161                    3
1   [10-20)    691                   40
2   [20-30)   1657                  236
3   [30-40)   3775                  424
4   [40-50)   9685                 1027
5   [50-60)  17256                 1668
6   [60-70)  22483                 2502
7   [70-80)  26068                 3069
8   [80-90)  17197                 2078
9  [90-100)   2793                  310


In [6]:
query3 = """
SELECT time_in_hospital, COUNT(*) as total,
SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) as readmitted_under_30,
ROUND(100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) / COUNT(*), 2) as pct_readmitted_under_30
FROM admissions
GROUP BY time_in_hospital
ORDER BY time_in_hospital
"""
print(pd.read_sql(query3, conn))

    time_in_hospital  total  readmitted_under_30  pct_readmitted_under_30
0                  1  14208                 1162                     8.18
1                  2  17224                 1712                     9.94
2                  3  17756                 1894                    10.67
3                  4  13924                 1644                    11.81
4                  5   9966                 1199                    12.03
5                  6   7539                  949                    12.59
6                  7   5859                  752                    12.83
7                  8   4391                  625                    14.23
8                  9   3002                  412                    13.72
9                 10   2342                  336                    14.35
10                11   1855                  195                    10.51
11                12   1448                  193                    13.33
12                13   1210           

In [7]:
query5 = """
SELECT race, COUNT(*) as total,
SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) as readmitted_under_30,
ROUND(100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) / COUNT(*), 2) as pct_readmitted_under_30
FROM admissions
GROUP BY race
ORDER BY total DESC
"""
print(pd.read_sql(query5, conn))

              race  total  readmitted_under_30  pct_readmitted_under_30
0        Caucasian  76099                 8592                    11.29
1  AfricanAmerican  19210                 2155                    11.22
2                ?   2273                  188                     8.27
3         Hispanic   2037                  212                    10.41
4            Other   1506                  145                     9.63
5            Asian    641                   65                    10.14


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Create a simple binary target: 1 if readmitted within 30 days, else 0
df['readmit_30'] = (df['readmitted'] == '<30').astype(int)

# Pick a few simple numeric features to start
features = ['time_in_hospital', 'num_lab_procedures', 'num_medications', 'number_diagnoses']
X = df[features]
y = df['readmit_30']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model accuracy:", model.score(X_test, y_test))
print("Feature importance (coefficients):", dict(zip(features, model.coef_[0])))

Model accuracy: 0.887737054141692
Feature importance (coefficients): {'time_in_hospital': np.float64(0.027875295687347174), 'num_lab_procedures': np.float64(0.0003331590599194827), 'num_medications': np.float64(0.005194144922364553), 'number_diagnoses': np.float64(0.07201757791555949)}


In [9]:
baseline_accuracy = 1 - y_test.mean()
print("Baseline accuracy (always predict 'not readmitted'):", baseline_accuracy)

Baseline accuracy (always predict 'not readmitted'): 0.887737054141692


In [10]:
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
model_balanced.fit(X_train, y_train)

print("Balanced model accuracy:", model_balanced.score(X_test, y_test))
print("Feature importance (coefficients):", dict(zip(features, model_balanced.coef_[0])))

from sklearn.metrics import recall_score, precision_score
y_pred = model_balanced.predict(X_test)
print("Recall (catching actual readmissions):", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))

Balanced model accuracy: 0.5020143460744817
Feature importance (coefficients): {'time_in_hospital': np.float64(0.029496320037013083), 'num_lab_procedures': np.float64(0.00034026466651494296), 'num_medications': np.float64(0.005342548812170777), 'number_diagnoses': np.float64(0.07316949474977459)}
Recall (catching actual readmissions): 0.587746170678337
Precision: 0.12745563253297904


In [11]:
# Replace '?' with actual missing value marker
import numpy as np
df = df.replace('?', np.nan)

# Check how much is missing in each column
print(df.isnull().sum().sort_values(ascending=False).head(10))

# Drop columns with very high missingness (not useful for analysis)
df_clean = df.drop(columns=['weight', 'payer_code', 'medical_specialty'])

# For race, drop rows with missing race rather than plotting 'unknown' as a category
df_clean = df_clean.dropna(subset=['race'])

print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(df_clean))

weight               98569
max_glu_serum        96420
A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
patient_nbr              0
dtype: int64
Rows before cleaning: 101766
Rows after cleaning: 99493
